
# **PARTE A: Evaluación de Modelos de Recomendación**


> Para el desarrollo y la optimización de este Notebook, se ha integrado la capacidad analítica de Gemini 3.1 Pro (la versión avanzada y actual de la familia Gemini) junto con el Modo IA de Google, herramientas que han permitido refinar la lógica del código, asegurar la precisión en el procesamiento de lenguaje natural y validar la coherencia semántica de las respuestas generadas.


Código en Python que cumple con los 8 puntos solicitados. Nos aseguramos de tener instalados `scikit-surprise`. El entorno donde hemos realizado las pruebas es Google Colab. Aquí se ha actualizado recientemente NumPy a su versión 2.0. Qué sucede? La librería `scikit-surprise` tiene componentes compilados en C que fueron construidos utilizando NumPy 1.x. Este cambio de versión mayor en NumPy rompe la compatibilidad binaria. Entoces vamos a bajar la versión del NumPy.

###0. Preparación del Entorno en el Notebook

In [ ]:
!pip install scikit-surprise "numpy<2"

  Using cached scikit_surprise-1.1.4.tar.gz (154 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554980 sha256=8569379f3714f265d24604d5b32ef4544ebdff9d54195faab5d8e213b5375a39
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of th

### 1. Importación de funciones

In [ ]:
# import pandas as pd
import os
import random
from surprise import Dataset, Reader, KNNBasic, SVD, NMF, accuracy
from surprise.model_selection import train_test_split
from surprise import dump
from collections import defaultdict

In [ ]:
# --- ANTES DE EMPEZAR ---
# Determinamos una variable SEMILLA para la reproducibilidad de los resultados
# por la profesora Marta
SEMILLA = 37

# --- 1. Carga el dataset de MovieLens de 100K ---
print("1. Cargando el dataset ml-100k...")
datos = Dataset.load_builtin('ml-100k')

# --- 2. Divide el dataset (75% entrenamiento, 25% prueba) ---
print("2. Dividiendo en conjunto de entrenamiento y evaluación...")
entrenamiento, prueba = train_test_split(datos, test_size=0.25, random_state=SEMILLA)

# --- 3 y 4. Definición y Entrenamiento de los algoritmos ---
print("3 y 4. Entrenando modelos (KNN, SVD, NMF)...")

# a. KNN basado en usuarios (Métrica: Pearson)
config_sim_usuario = {'name': 'pearson', 'user_based': True}
knn_usuario = KNNBasic(sim_options=config_sim_usuario)
knn_usuario.fit(entrenamiento) # KNNBasic no recibe random_state por diseño de la librería

# a. KNN basado en productos (Métrica: Pearson)
config_sim_producto = {'name': 'pearson', 'user_based': False}
knn_producto = KNNBasic(sim_options=config_sim_producto)
knn_producto.fit(entrenamiento)

# b. Factorización de matrices: SVD
svd = SVD(random_state=SEMILLA)
svd.fit(entrenamiento)

# b. Factorización de matrices: NMF
nmf = NMF(random_state=SEMILLA)
nmf.fit(entrenamiento)

# --- 5. Obtener predicciones e interpretar ---
print("\n5. Obteniendo predicciones del conjunto de evaluación...")
preds_knn_usuario = knn_usuario.test(prueba)
preds_knn_producto = knn_producto.test(prueba)
preds_svd = svd.test(prueba)
preds_nmf = nmf.test(prueba)

print("Muestra de 5 predicciones (usando SVD):")
for i in range(5):
    p = preds_svd[i]
    print(f"Usuario: {p.uid} | Producto (Película): {p.iid} | Calificación Real: {p.r_ui} | Calificación Estimada: {p.est:.2f}")

print("")
print("Muestra de 5 predicciones (usando NMF):")
for i in range(5):
    p = preds_nmf[i]
    print(f"Usuario: {p.uid} | Producto (Película): {p.iid} | Calificación Real: {p.r_ui} | Calificación Estimada: {p.est:.2f}")

print("")
print("Muestra de 5 predicciones (usando KNN basado en usuarios):")
for i in range(5):
    p = preds_knn_usuario[i]
    print(f"Usuario: {p.uid} | Producto (Película): {p.iid} | Calificación Real: {p.r_ui} | Calificación Estimada: {p.est:.2f}")

print("")
print("Muestra de 5 predicciones (usando KNN basado en productos):")
for i in range(5):
    p = preds_knn_producto[i]
    print(f"Usuario: {p.uid} | Producto (Película): {p.iid} | Calificación Real: {p.r_ui} | Calificación Estimada: {p.est:.2f}")

1. Cargando el dataset ml-100k...
Dataset ml-100k could not be found. Do you want to download it? [Y/n] y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
2. Dividiendo en conjunto de entrenamiento y evaluación...
3 y 4. Entrenando modelos (KNN, SVD, NMF)...
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.

5. Obteniendo predicciones del conjunto de evaluación...
Muestra de 5 predicciones (usando SVD):
Usuario: 747 | Producto (Película): 116 | Calificación Real: 4.0 | Calificación Estimada: 4.13
Usuario: 65 | Producto (Película): 378 | Calificación Real: 5.0 | Calificación Estimada: 3.93
Usuario: 521 | Producto (Película): 147 | Calificación Real: 4.0 | Calificación Estimada: 3.00
Usuario: 160 | Producto (Película): 462 | Calificación Real: 4.0 | Calificación Estimada: 3

**Interpretación del paso 5:** En las predicciones vemos el ID del usuario (`uid`), el ID de la película (`iid`), el rating real que el usuario le dio (`r_ui`) y la predicción que hace nuestro modelo (`est`). La diferencia entre el Calificación Real y el Estimado es el "error" de nuestra predicción.

In [ ]:
# --- 6. Tabla con los valores RMSE ---
print("\n6. Calculando y creando tabla de métricas RMSE...")
rmse_knn_usuario = accuracy.rmse(preds_knn_usuario, verbose=False)
rmse_knn_producto = accuracy.rmse(preds_knn_producto, verbose=False)
rmse_svd = accuracy.rmse(preds_svd, verbose=False)
rmse_nmf = accuracy.rmse(preds_nmf, verbose=False)

'''
df_resultados = pd.DataFrame({
    'Algoritmo': ['KNN Basado-usuario (Pearson)', 'KNN Basado-producto (Pearson)', 'SVD', 'NMF'],
    'RMSE': [rmse_knn_usuario, rmse_knn_producto, rmse_svd, rmse_nmf]
}).sort_values(by='RMSE')

print("\n", df_resultados.to_string(index=False))
'''

# a. Creamos el diccionario (Clave: Algoritmo, Valor: RMSE)
resultados = {
    'KNN Basado-usuario (Pearson)': rmse_knn_usuario,
    'KNN Basado-producto (Pearson)': rmse_knn_producto,
    'SVD': rmse_svd,
    'NMF': rmse_nmf
}

# b. Ordenamos por el valor (el RMSE) de menor a mayor
# sorted devuelve una lista de tuplas: [('Algoritmo', valor), ...]
resultados_ordenados = sorted(resultados.items(), key=lambda item: item[1])

# c. Imprimimos el "ranking"
print(f"\n{'Algoritmo':<30} | {'RMSE'}")
print("-" * 40)
for algoritmo, rmse in resultados_ordenados:
    print(f"{algoritmo:<30} | {rmse:.4f}")



6. Calculando y creando tabla de métricas RMSE...

Algoritmo                      | RMSE
----------------------------------------
SVD                            | 0.9351
NMF                            | 0.9649
KNN Basado-usuario (Pearson)   | 1.0123
KNN Basado-producto (Pearson)  | 1.0433


### 7. Explicación de los resultados de RMSE
El RMSE (Root Mean Square Error - Error Cuadrático Medio) mide cuánto se desvían, en promedio, las calificaciones estimadas por el modelo de las calificaciones reales del usuario. Como está en la misma escala que los ratings (de 1 a 5), un RMSE de 0.94 significa que nuestras predicciones fallan en promedio por casi 1 estrella.
**El mejor método recomendador** es aquel con el **RMSE más bajo**. En sistemas de recomendación con datasets densos como MovieLens, el algoritmo **SVD** suele ser el ganador sistemáticamente (gracias a su capacidad para capturar características latentes). Seleccionaremos SVD como nuestro mejor modelo.

In [ ]:
# --- 8. Recomendar 10 películas al azar con el mejor modelo (SVD) ---
print("\n8. Generando top 10 recomendaciones para un usuario aleatorio...")

# Agrupar las predicciones de SVD por usuario
predicciones_usuario = defaultdict(list)
for uid, iid, true_r, est, _ in preds_svd:
    predicciones_usuario[uid].append((iid, est))

# Ordenar las predicciones por la calificación estimado para cada usuario
for uid, calificacion_usuario in predicciones_usuario.items():
    calificacion_usuario.sort(key=lambda x: x[1], reverse=True)
    predicciones_usuario[uid] = calificacion_usuario[:10]

random.seed(SEMILLA)
aleatorio_usuario = random.choice(list(predicciones_usuario.keys()))

print(f"\nTop 10 recomendaciones para el usuario {aleatorio_usuario}:")
for rank, (iid, est) in enumerate(predicciones_usuario[aleatorio_usuario], 1):
    print(f" {rank}. Película ID: {iid} | Calificación esperada: {est:.2f}")

# --- GUARDAR EL MODELO PARA LA APP ---
print("\nGuardando el modelo SVD para la app cliente-servidor...")
dump.dump('mejor_modelo_svd.pkl', algo=svd)
print("Modelo guardado como 'mejor_modelo_svd.pkl'")


8. Generando top 10 recomendaciones para un usuario aleatorio...

Top 10 recomendaciones para el usuario 376:
 1. Película ID: 705 | Calificación esperada: 4.02
 2. Película ID: 663 | Calificación esperada: 3.89
 3. Película ID: 288 | Calificación esperada: 3.50

Guardando el modelo SVD para la app cliente-servidor...
Modelo guardado como 'mejor_modelo_svd.pkl'


Como comentamos, el problema del enfoque anterior es que predecía sobre el conjunto prueba (test), el cual contiene películas que el usuario ya vio (por eso tenemos un `r_ui` o calificación real con la que comparar el error).

Para hacer recomendaciones reales, necesitamos que el modelo evalúe las películas que el usuario no ha visto. Para esto, `scikit-surprise` tiene el método `build_anti_testset()`, que genera exactamente eso: todos los pares usuario-película que faltan en el entrenamiento.

In [ ]:
# --- 8. Recomendar 10 películas NUEVAS con el mejor modelo (SVD) ---
print("\n8. Generando top 10 recomendaciones de películas NO VISTAS para un usuario aleatorio...")

# Obtener la lista de todos los usuarios (IDs reales) en el set de entrenamiento
usuarios_existentes = [entrenamiento.to_raw_uid(u) for u in entrenamiento.all_users()]

# Elegir un usuario al azar asegurando la reproducibilidad
random.seed(SEMILLA)
aleatorio_usuario = random.choice(usuarios_existentes)

# Construir el 'anti-testset'
# Esto genera los pares (usuario, película) que no tienen calificación.
# OJO: Hacerlo para todos los usuarios consume mucha memoria, así que le pasamos
# un fill_value (por defecto es la media global) pero luego lo filtraremos.
anti_prueba_completo = entrenamiento.build_anti_testset()

# Filtrar el anti-testset SOLO para el usuario elegido (para que sea súper rápido)
anti_prueba_usuario = [par for par in anti_prueba_completo if par[0] == aleatorio_usuario]

# Obtener las predicciones del modelo SVD para esas películas no vistas
predicciones_usuario = svd.test(anti_prueba_usuario)

# Ordenar las predicciones por la calificación estimada (est) de mayor a menor
predicciones_usuario.sort(key=lambda x: x.est, reverse=True)

# Mostrar el Top 10
print(f"\nTop 10 recomendaciones (películas nuevas) para el usuario {aleatorio_usuario}:")
for rank, prediccion in enumerate(predicciones_usuario[:10], 1):
    print(f" {rank}. Película ID: {prediccion.iid} | Calificación esperada: {prediccion.est:.2f}")

# --- GUARDAR EL MODELO PARA LA APP ---
print("\nGuardando el modelo SVD para la app cliente-servidor...")
dump.dump('mejor_modelo_svd.pkl', algo=svd)
print("Modelo guardado como 'mejor_modelo_svd.pkl'")


8. Generando top 10 recomendaciones de películas NO VISTAS para un usuario aleatorio...

Top 10 recomendaciones (películas nuevas) para el usuario 767:
 1. Película ID: 169 | Calificación esperada: 5.00
 2. Película ID: 357 | Calificación esperada: 4.91
 3. Película ID: 64 | Calificación esperada: 4.88
 4. Película ID: 603 | Calificación esperada: 4.88
 5. Película ID: 480 | Calificación esperada: 4.87
 6. Película ID: 136 | Calificación esperada: 4.87
 7. Película ID: 12 | Calificación esperada: 4.86
 8. Película ID: 114 | Calificación esperada: 4.85
 9. Película ID: 48 | Calificación esperada: 4.83
 10. Película ID: 127 | Calificación esperada: 4.83

Guardando el modelo SVD para la app cliente-servidor...
Modelo guardado como 'mejor_modelo_svd.pkl'


Mostrando el título real que da un aspecto mucho más profesional y terminado a nuestro sistema de recomendación.

**Crear el diccionario de películas:**
Añadimos este bloque antes de generar el Top 10. Es importante utilizar la codificación `iso-8859-1`, ya que este dataset antiguo tiene algunos caracteres especiales que fallarían con el clásico `utf-8`.

In [ ]:
# Ruta donde scikit-surprise guarda los datos por defecto
ruta_productos = os.path.expanduser('~/.surprise_data/ml-100k/ml-100k/u.item')

print("Cargando los nombres de las películas...")
mapa_peliculas = {}

# Leemos el archivo u.item
with open(ruta_productos, 'r', encoding='iso-8859-1') as f:
    for linea in f:
        # El archivo ml-100k separa los campos con la barra vertical '|'
        datos = linea.split('|')
        id_pelicula = datos[0]
        titulo_pelicula = datos[1]
        mapa_peliculas[id_pelicula] = titulo_pelicula



# Mostramos el Top 10 con los TÍTULOS reales
print(f"\nTop 10 recomendaciones (películas nuevas) para el usuario {aleatorio_usuario}:")

for rank, prediccion in enumerate(predicciones_usuario[:10], 1):
    id_peli = prediccion.iid
    # Buscamos el título en nuestro diccionario. Si no está, colocamos "Desconocido"
    titulo = mapa_peliculas.get(id_peli, "Título desconocido")

    print(f" {rank}. {titulo} (ID: {id_peli}) | Calificación esperada: {prediccion.est:.2f}")

Cargando los nombres de las películas...

Top 10 recomendaciones (películas nuevas) para el usuario 767:
 1. Wrong Trousers, The (1993) (ID: 169) | Calificación esperada: 5.00
 2. One Flew Over the Cuckoo's Nest (1975) (ID: 357) | Calificación esperada: 4.91
 3. Shawshank Redemption, The (1994) (ID: 64) | Calificación esperada: 4.88
 4. Rear Window (1954) (ID: 603) | Calificación esperada: 4.88
 5. North by Northwest (1959) (ID: 480) | Calificación esperada: 4.87
 6. Mr. Smith Goes to Washington (1939) (ID: 136) | Calificación esperada: 4.87
 7. Usual Suspects, The (1995) (ID: 12) | Calificación esperada: 4.86
 8. Wallace & Gromit: The Best of Aardman Animation (1996) (ID: 114) | Calificación esperada: 4.85
 9. Hoop Dreams (1994) (ID: 48) | Calificación esperada: 4.83
 10. Godfather, The (1972) (ID: 127) | Calificación esperada: 4.83


### Conclusión

**I\. Superioridad absoluta de la Factorización de Matrices (SVD)**

El resultado más claro de la ejecución es que el algoritmo **SVD** es el ganador indiscutible con un RMSE de 0.9351. Esto demuestra que, para datasets de este tipo (MovieLens), comprimir el historial en "factores latentes" funciona mucho mejor que la simple búsqueda de vecinos. SVD es capaz de descubrir patrones ocultos (por ejemplo, que a un usuario le gustan las películas de los años 80 dirigidas por un director específico) que los métodos tradicionales no pueden ver.

**II\. El error del modelo es muy aceptable para producción (Significado del RMSE)**

Al analizar el RMSE ganador de 0.9351, la conclusión de negocio es muy positiva. Al tener en cuenta que las valoraciones van del 1 al 5, esto significa que nuestro modelo se equivoca, en promedio, por **menos de 1 estrella** al predecir el gusto de un usuario. Es decir, si el sistema dice que a alguien le va a encantar una película (le da un 4.5), en el peor de los casos el usuario le daría un 3.5 (le parecerá "bien"). Esto garantiza una buena experiencia de usuario.

**III\. Los algoritmos basados en memoria (KNN) sufren en este entorno**

Los modelos KNN (tanto basados en usuario con un RMSE de 1.0123, como en producto con 1.0433) quedaron en los últimos puestos. La conclusión aquí es que estos algoritmos basados en la similitud directa sufren mucho con la "esparcidad" de los datos (la mayoría de los usuarios no han visto la mayoría de las películas). Además, curiosamente en nuestras pruebas de partición de datos, buscar usuarios similares funcionó ligeramente mejor que buscar películas similares.

**IV\. Robustez y preparación para el despliegue**

Más allá de las métricas puras, el código demostró que el pipeline es sólido. Al establecer una semilla (SEMILLA \= 37), garantizamos que nuestros experimentos son 100% reproducibles para cualquier auditoría. Además, al concluir exportando el modelo ganador (mejor\_modelo\_svd.pkl), demostramos que el trabajo no se queda en un entorno académico, sino que está listo para ser inyectado en una aplicación real cliente-servidor.  